# Kronos Forecaster + Jev

Free candlestick forecasting with **Kronos** (the open-source model from the reel), plus a second opinion from **Jev** (TypeSafe AI's decision model). Wrapped so you never touch code.

**How to use**
1. Click **Runtime → Run all** in the top menu. The first run takes about a minute.
2. Scroll down. Step 3 shows the chart and a plain-English summary. Step 4 shows how the model would have done on the last stretch. Step 5 asks Jev for a second opinion.
3. To try another ticker, change the box in **Step 2**, then click the play button on Steps 2 to 5.

Tickers that work: stocks `AAPL`, `NVDA`, `TSLA`; funds `SPY`, `QQQ`; crypto `BTC-USD`, `ETH-USD`, `SOL-USD`.

**Step 5 needs your Vercel AI Gateway key, once.** Click the key icon in the left sidebar (Secrets), add a secret named `AI_GATEWAY_API_KEY`, paste the key, switch on Notebook access. Without it, Step 5 asks you to paste the key each session.

**Faster (optional):** Runtime → Change runtime type → **T4 GPU** → Save. Free tier.


In [ ]:
#@title 1) Setup  (run once per session, about 1 minute)
import subprocess, sys, os
def sh(cmd):
    subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
try:
    import torch
except ImportError:
    sh(f"{sys.executable} -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu")
    import torch
sh(f"{sys.executable} -m pip install -q einops huggingface_hub safetensors yfinance matplotlib pandas numpy tqdm requests")
if not os.path.isdir("Kronos"):
    sh("git clone -q --depth 1 https://github.com/shiyu-coder/Kronos.git")
open("kronos_forecast.py", "w").write(r'''"""Kronos Forecaster helper: fetch a price chart for any ticker, run Kronos, return a probabilistic forecast."""
import sys, os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

KRONOS_DIR = os.environ.get("KRONOS_DIR", "Kronos")
if KRONOS_DIR not in sys.path:
    sys.path.insert(0, KRONOS_DIR)
from model import Kronos, KronosTokenizer, KronosPredictor  # noqa: E402

MODEL_IDS = {"mini": "NeoQuasar/Kronos-mini", "small": "NeoQuasar/Kronos-small", "base": "NeoQuasar/Kronos-base"}
TOKENIZER_IDS = {"mini": "NeoQuasar/Kronos-Tokenizer-2k", "small": "NeoQuasar/Kronos-Tokenizer-base", "base": "NeoQuasar/Kronos-Tokenizer-base"}
MAX_CONTEXT = {"mini": 2048, "small": 512, "base": 512}
X_COLS = ["open", "high", "low", "close", "volume", "amount"]


def load_predictor(size="small"):
    tok = KronosTokenizer.from_pretrained(TOKENIZER_IDS[size])
    mdl = Kronos.from_pretrained(MODEL_IDS[size])
    return KronosPredictor(mdl, tok, max_context=MAX_CONTEXT[size])


def is_crypto(ticker):
    return ticker.upper().endswith(("-USD", "-USDT", "-BTC", "-EUR"))


def fetch(ticker, lookback=400):
    """Daily candles from Yahoo Finance. Works for stocks (AAPL), ETFs (SPY), crypto (BTC-USD), FX (EURUSD=X)."""
    import yfinance as yf
    raw = yf.download(ticker, period="5y", interval="1d", auto_adjust=False, progress=False)
    if raw is None or len(raw) == 0:
        raise ValueError(f"No data for '{ticker}'. Check the symbol (examples: AAPL, NVDA, BTC-USD, ETH-USD, SPY).")
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    idx = pd.to_datetime(raw.index)
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_localize(None)
    df = pd.DataFrame({
        "timestamps": idx,
        "open": raw["Open"].values.astype(float), "high": raw["High"].values.astype(float),
        "low": raw["Low"].values.astype(float), "close": raw["Close"].values.astype(float),
        "volume": raw["Volume"].values.astype(float),
    }).dropna().reset_index(drop=True)
    df["amount"] = df["volume"] * df[["open", "high", "low", "close"]].mean(axis=1)
    # Drop today's candle if it is still forming (partial volume would skew the model's view).
    today = pd.Timestamp.utcnow().tz_localize(None).normalize()
    if df["timestamps"].iloc[-1].normalize() >= today:
        df = df.iloc[:-1]
        print(f"  (skipped today's unfinished candle; forecasting from {df['timestamps'].iloc[-1].date()})")
    if len(df) < 60:
        raise ValueError(f"Only {len(df)} days of data for '{ticker}'. Need at least 60.")
    return df.tail(lookback).reset_index(drop=True)


def future_dates(last_ts, n, crypto):
    start = last_ts + pd.Timedelta(days=1)
    idx = pd.date_range(start, periods=n, freq="D") if crypto else pd.bdate_range(start, periods=n)
    return pd.Series(idx)


def forecast_paths(predictor, df, days_ahead=30, n_paths=20, temperature=1.0, top_p=0.9, crypto=False, chunk=5):
    """Run Kronos n_paths times in small batches (memory-safe). Returns (paths[n, days, 6], future_timestamps)."""
    x_ts = df["timestamps"]
    y_ts = future_dates(x_ts.iloc[-1], days_ahead, crypto)
    out = []
    done = 0
    while done < n_paths:
        b = min(chunk, n_paths - done)
        preds = predictor.predict_batch(
            df_list=[df[X_COLS]] * b, x_timestamp_list=[x_ts] * b, y_timestamp_list=[y_ts] * b,
            pred_len=days_ahead, T=temperature, top_p=top_p, sample_count=1, verbose=False)
        out.extend(p[X_COLS].values for p in preds)
        done += b
        print(f"  simulated {done}/{n_paths} paths", end="\r")
    print()
    return np.stack(out), y_ts


def summarize(df, paths, y_ts, ticker):
    closes = paths[:, :, 3]                      # close column -> (n_paths, days)
    last = float(df["close"].iloc[-1])
    end = closes[:, -1]
    med_path = np.median(closes, axis=0)
    s = {
        "ticker": ticker, "last_close": last, "horizon_days": closes.shape[1], "paths": closes.shape[0],
        "median_end": float(np.median(end)),
        "p10_end": float(np.percentile(end, 10)), "p90_end": float(np.percentile(end, 90)),
        "prob_up": float((end > last).mean()),
        "median_path": med_path,
        "p10": np.percentile(closes, 10, axis=0), "p90": np.percentile(closes, 90, axis=0),
        "p25": np.percentile(closes, 25, axis=0), "p75": np.percentile(closes, 75, axis=0),
        "dates": y_ts,
    }
    s["median_change"] = s["median_end"] / last - 1
    s["p10_change"] = s["p10_end"] / last - 1
    s["p90_change"] = s["p90_end"] / last - 1
    return s


def fmt_money(v):
    if v >= 1e5:
        return f"${v:,.0f}"
    if v >= 1:
        return f"${v:,.2f}"
    return f"${v:,.4f}"


def plain_english(s):
    d = s["horizon_days"]
    return "\n".join([
        f"{s['ticker']}  |  last close {fmt_money(s['last_close'])}  |  {d} trading days ahead  |  {s['paths']} simulated paths",
        "",
        f"Middle-of-the-road outcome: {fmt_money(s['median_end'])} ({s['median_change']:+.1%})",
        f"80% of paths finish between {fmt_money(s['p10_end'])} ({s['p10_change']:+.1%}) and {fmt_money(s['p90_end'])} ({s['p90_change']:+.1%})",
        f"Share of paths that end higher than today: {s['prob_up']:.0%}",
        "",
        "Read this like a weather forecast, not a promise. A 60% 'higher' means 40% of the time it isn't.",
    ])


def plot_forecast(df, s, history_days=120, actual=None, title=None):
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    hist = df.tail(history_days)
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.plot(hist["timestamps"], hist["close"], color="#1f2937", lw=1.6, label="Actual (history)")
    x = pd.to_datetime(s["dates"])
    x0 = pd.concat([pd.Series([hist["timestamps"].iloc[-1]]), pd.Series(x)]).values
    l0 = s["last_close"]
    ax.fill_between(x0, np.r_[l0, s["p10"]], np.r_[l0, s["p90"]], color="#3b82f6", alpha=0.18, label="80% range")
    ax.fill_between(x0, np.r_[l0, s["p25"]], np.r_[l0, s["p75"]], color="#3b82f6", alpha=0.32, label="50% range")
    ax.plot(x0, np.r_[l0, s["median_path"]], color="#1d4ed8", lw=2.0, label="Median forecast")
    if actual is not None:
        ax.plot(actual["timestamps"], actual["close"], color="#dc2626", lw=1.8, ls="--", label="What actually happened")
    ax.axvline(hist["timestamps"].iloc[-1], color="#9ca3af", lw=1, ls=":")
    ax.set_title(title or f"{s['ticker']}: Kronos forecast, next {s['horizon_days']} trading days", fontsize=13, loc="left")
    ax.set_ylabel("Close price"); ax.grid(alpha=0.25); ax.legend(loc="upper left", frameon=False)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    return fig


def backtest(predictor, df, days_ahead=30, n_paths=20, crypto=False, temperature=1.0, top_p=0.9, chunk=5):
    """Hide the last `days_ahead` bars, forecast them, and score against what really happened."""
    train = df.iloc[:-days_ahead].reset_index(drop=True)
    actual = df.iloc[-days_ahead:].reset_index(drop=True)
    paths, _ = forecast_paths(predictor, train, days_ahead, n_paths, temperature, top_p, crypto, chunk)
    s = summarize(train, paths, actual["timestamps"], "Reality check")
    a_end = float(actual["close"].iloc[-1]); last = s["last_close"]
    inside = (actual["close"].values >= s["p10"]) & (actual["close"].values <= s["p90"])
    res = {
        "direction_right": bool((a_end > last) == (s["median_end"] > last)),
        "actual_change": a_end / last - 1, "median_change": s["median_change"],
        "end_inside_80": bool(s["p10_end"] <= a_end <= s["p90_end"]),
        "days_inside_80": float(inside.mean()),
        "mape_median": float(np.mean(np.abs(s["median_path"] - actual["close"].values) / actual["close"].values)),
    }
    return s, actual, train, res


def backtest_english(res, days):
    return "\n".join([
        f"Reality check: the last {days} trading days were hidden from the model, then forecast.",
        f"  Direction call: {'RIGHT' if res['direction_right'] else 'WRONG'}  (model said {res['median_change']:+.1%}, market did {res['actual_change']:+.1%})",
        f"  Final price landed inside the 80% range: {'YES' if res['end_inside_80'] else 'NO'}",
        f"  Days the real price stayed inside the 80% range: {res['days_inside_80']:.0%}",
        f"  Average miss of the median line: {res['mape_median']:.1%}",
        "",
        "One reality check is one coin flip. Run it on a few tickers before trusting the pattern.",
    ])
''')
open("jev_judge.py", "w").write(r'''"""Jev judgment layer for the Kronos Forecaster.

Hands the Kronos forecast plus simple price statistics to Jev, TypeSafe AI's System One decision model,
and gets back typed, calibrated answers. Jev judges; Kronos forecasts; code executes.

Access route: Vercel AI Gateway's TypeSafe-compatible API, exactly as documented by TypeSafe
(docs.typesafe.ai/sdk/python/usage, "Configuring the base URL", Vercel tab):
    base_url = "https://ai-gateway.vercel.sh/typesafe", model = "typesafe-ai/jev"
Request/response shapes follow docs.typesafe.ai/api (POST /v1/systemone).
"""
import os
import numpy as np
import pandas as pd
import requests

JEV_ENDPOINT = "https://ai-gateway.vercel.sh/typesafe/v1/systemone"   # official Vercel AI Gateway route
JEV_MODEL = "typesafe-ai/jev"
FLAT_BAND_PCT = 2.0


def get_gateway_key():
    """Vercel AI Gateway key: environment variable, then Colab Secrets, then a hidden paste prompt."""
    key = os.environ.get("AI_GATEWAY_API_KEY")
    if not key:
        try:
            from google.colab import userdata  # type: ignore
            key = userdata.get("AI_GATEWAY_API_KEY")
        except Exception:
            key = None
    if not key:
        import getpass
        key = getpass.getpass("Paste your Vercel AI Gateway key (hidden): ").strip()
    return key


def _pct(a, b):
    return float((a / b - 1.0) * 100.0)


def build_state(df, summary, score=None):
    """Compact, numbers-only snapshot. Jev sees exactly this and nothing else."""
    c = df["close"].astype(float).values
    last = float(c[-1])
    rets = np.diff(np.log(c[-21:]))
    vol_20 = float(np.std(rets) * np.sqrt(252) * 100) if len(rets) > 2 else None
    window = c[-60:]
    peak = np.maximum.accumulate(window)
    max_dd_60 = float(((window / peak) - 1.0).min() * 100)
    state = {
        "ticker": summary["ticker"],
        "as_of": str(pd.Timestamp(df["timestamps"].iloc[-1]).date()),
        "horizon_trading_days": int(summary["horizon_days"]),
        "flat_band_pct": FLAT_BAND_PCT,
        "price": {
            "last_close": round(last, 4),
            "change_5d_pct": round(_pct(last, c[-6]), 2) if len(c) > 6 else None,
            "change_20d_pct": round(_pct(last, c[-21]), 2) if len(c) > 21 else None,
            "change_60d_pct": round(_pct(last, c[-61]), 2) if len(c) > 61 else None,
            "vs_50d_average_pct": round(_pct(last, c[-50:].mean()), 2) if len(c) >= 50 else None,
            "vs_200d_average_pct": round(_pct(last, c[-200:].mean()), 2) if len(c) >= 200 else None,
            "volatility_20d_annualized_pct": round(vol_20, 1) if vol_20 is not None else None,
            "max_drawdown_60d_pct": round(max_dd_60, 2),
            "last_10_closes": [round(float(v), 4) for v in c[-10:]],
        },
        "kronos_forecast": {
            "model": f"Kronos candlestick foundation model, {summary['paths']} simulated paths",
            "median_change_pct": round(summary["median_change"] * 100, 2),
            "p10_change_pct": round(summary["p10_change"] * 100, 2),
            "p90_change_pct": round(summary["p90_change"] * 100, 2),
            "share_of_paths_ending_higher": round(summary["prob_up"], 2),
        },
    }
    if score is not None:
        state["kronos_reality_check_on_previous_period"] = {
            "direction_correct": bool(score["direction_right"]),
            "model_said_pct": round(score["median_change"] * 100, 2),
            "market_did_pct": round(score["actual_change"] * 100, 2),
            "final_price_inside_80pct_band": bool(score["end_inside_80"]),
            "share_of_days_inside_band": round(score["days_inside_80"], 2),
            "avg_miss_of_median_pct": round(score["mape_median"] * 100, 2),
        }
    return state


def build_questions(horizon):
    return {
        "direction": {
            "type": "choice",
            "instructions": (
                f"Over the next {horizon} trading days, which outcome is most likely for `ticker`? "
                "Weigh the `price` statistics, the `kronos_forecast`, and, if present, how Kronos did on the "
                "`kronos_reality_check_on_previous_period`."
            ),
            "criteria": {
                "up": f"Closes more than {FLAT_BAND_PCT}% above `price.last_close`",
                "flat": f"Closes within {FLAT_BAND_PCT}% of `price.last_close`",
                "down": f"Closes more than {FLAT_BAND_PCT}% below `price.last_close`",
            },
        },
        "trust": {
            "type": "score",
            "instructions": "How much weight should a cautious investor give this Kronos forecast?",
            "criteria": [
                "Ignore it: the reality check failed badly or the 80% range is too wide to act on",
                "Weak hint: directionally plausible, but the range is wide or the reality check was mixed",
                "Worth watching: consistent with recent price action and a decent reality check",
                "Strong signal: tight range, good reality check, and price action clearly agrees",
            ],
        },
        "regime": {
            "type": "choice",
            "instructions": "Which best describes the current price regime in `price`?",
            "criteria": {
                "trending_up": "Steady gains, above its averages, moderate volatility",
                "trending_down": "Steady losses, below its averages, moderate volatility",
                "sideways_choppy": "No clear direction, mean-reverting moves",
                "volatile_shock": "Unusually large recent moves or drawdown; volatility elevated",
            },
        },
        "agrees": {
            "type": "noul",
            "instructions": "Does the recent price action in `price` support the direction of `kronos_forecast.median_change_pct`?",
            "criteria": {"true": "Momentum and trend point the same way as the forecast",
                         "false": "They disagree or the forecast is essentially flat"},
        },
    }


def ask_jev(state, questions, api_key, timeout=60):
    payload = {"model": JEV_MODEL, "state": state, "questions": questions}
    r = requests.post(JEV_ENDPOINT, json=payload, timeout=timeout,
                      headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"})
    if r.status_code != 200:
        body = r.text[:400]
        if "customer_verification_required" in body:
            raise RuntimeError("Vercel needs a payment card on the team before AI Gateway works (it stays free: $5 of "
                               "monthly credit). Add a card at vercel.com -> Settings -> Billing, then run this cell again.")
        if r.status_code in (401, 403):
            raise RuntimeError(f"The AI Gateway key was rejected ({r.status_code}). Check the key and try again. Details: {body}")
        raise RuntimeError(f"Jev call failed with HTTP {r.status_code}: {body}")
    return r.json()


def jev_verdict(df, summary, score=None, api_key=None):
    api_key = api_key or get_gateway_key()
    state = build_state(df, summary, score)
    out = ask_jev(state, build_questions(summary["horizon_days"]), api_key)
    return out, state


def _bar(p):
    n = int(round(p * 20))
    return "#" * n + "." * (20 - n)


def verdict_english(out, summary):
    a = out["answers"]
    d, t, r, g = a["direction"], a["trust"], a["regime"], a["agrees"]
    probs = d["probabilities"]
    trust_level = t["legend"][str(int(round(t["score"])))].split(":")[0]
    lines = [
        f"Jev's verdict on {summary['ticker']}, next {summary['horizon_days']} trading days   (model: {out.get('model', '?')})",
        "",
        f"  Direction call: {d['choice'].upper()}   confidence {d['confidence']:.2f}",
    ]
    for k in ("up", "flat", "down"):
        lines.append(f"     {k:<5} {_bar(probs.get(k, 0))} {probs.get(k, 0):.0%}")
    lines += [
        f"  Trust in the Kronos forecast: {trust_level}  ({t['score']:.1f} on a 0-3 scale, confidence {t['confidence']:.2f})",
        f"  Price regime: {r['choice'].replace('_', ' ')}  ({r['probabilities'][r['choice']]:.0%})",
        f"  Price action agrees with Kronos: {g['noul']:.0%} yes",
        "",
        "  Jev judges, Kronos forecasts, you decide. Jev only sees the numbers in the snapshot: no news, no earnings, no macro.",
        f"  Cost of this call: about {out.get('usage', {}).get('input_tokens', 0)} tokens (fractions of a cent).",
    ]
    return "\n".join(lines)
''')
import importlib, kronos_forecast as kf, jev_judge as jj
importlib.reload(kf); importlib.reload(jj)
_predictor_cache = {}
print("Ready.  Device:", "GPU (fast)" if torch.cuda.is_available() else "CPU (works fine, a little slower)")


In [ ]:
#@title 2) Settings  (change these, then run 3)
TICKER = "BTC-USD"  #@param {type:"string"}
DAYS_AHEAD = 30  #@param {type:"slider", min:5, max:90, step:5}
PATHS = 20  #@param {type:"slider", min:5, max:60, step:5}
MODEL_SIZE = "small"  #@param ["small", "base", "mini"]
print(f"Ticker {TICKER}  |  {DAYS_AHEAD} trading days ahead  |  {PATHS} paths  |  Kronos-{MODEL_SIZE}")


In [ ]:
#@title 3) Run the forecast
import matplotlib.pyplot as plt
import kronos_forecast as kf
if MODEL_SIZE not in _predictor_cache:
    print(f"Loading Kronos-{MODEL_SIZE} from Hugging Face (first time only)...")
    _predictor_cache[MODEL_SIZE] = kf.load_predictor(MODEL_SIZE)
predictor = _predictor_cache[MODEL_SIZE]
df = kf.fetch(TICKER, lookback=400)
print(f"Got {len(df)} days of {TICKER}, last close {kf.fmt_money(df['close'].iloc[-1])} on {df['timestamps'].iloc[-1].date()}")
paths, future = kf.forecast_paths(predictor, df, DAYS_AHEAD, PATHS, crypto=kf.is_crypto(TICKER))
summary = kf.summarize(df, paths, future, TICKER)
print(kf.plain_english(summary))
kf.plot_forecast(df, summary)
plt.show()


In [ ]:
#@title 4) Reality check  (how would it have done on the LAST stretch?)
bt_summary, actual, train, score = kf.backtest(predictor, df, DAYS_AHEAD, PATHS, crypto=kf.is_crypto(TICKER))
print(kf.backtest_english(score, DAYS_AHEAD))
kf.plot_forecast(train, bt_summary, actual=actual,
                 title=f"{TICKER}: what Kronos would have said {DAYS_AHEAD} trading days ago (red dashed = reality)")
plt.show()


In [ ]:
#@title 5) Ask Jev  (second opinion from TypeSafe AI's decision model, via your Vercel AI Gateway)
import jev_judge as jj
try:
    score
except NameError:
    score = None   # run Step 4 first if you want Jev to see the reality check
verdict, snapshot = jj.jev_verdict(df, summary, score)
print(jj.verdict_english(verdict, summary))


In [ ]:
#@title 6) Optional: scan a whole watchlist
WATCHLIST = "AAPL, NVDA, TSLA, SPY, BTC-USD, ETH-USD"  #@param {type:"string"}
import pandas as pd
rows = []
for t in [x.strip().upper() for x in WATCHLIST.split(",") if x.strip()]:
    try:
        d = kf.fetch(t, 400)
        p, fut = kf.forecast_paths(predictor, d, DAYS_AHEAD, PATHS, crypto=kf.is_crypto(t))
        ss = kf.summarize(d, p, fut, t)
        rows.append({"ticker": t, "last close": kf.fmt_money(ss["last_close"]),
                     "median": f"{ss['median_change']:+.1%}", "80% low": f"{ss['p10_change']:+.1%}",
                     "80% high": f"{ss['p90_change']:+.1%}", "paths ending higher": f"{ss['prob_up']:.0%}"})
    except Exception as e:
        rows.append({"ticker": t, "last close": "error", "median": str(e)[:70]})
table = pd.DataFrame(rows)
try:
    display(table)
except NameError:
    print(table.to_string(index=False))


## How to read the chart

- **Dark line**: real prices up to today.
- **Blue line**: the middle of all simulated paths (the median).
- **Dark blue band**: half the paths land here. **Light blue band**: 80% of paths land here.
- **Step 4, red dashed line**: what really happened, drawn over what the model would have said.

## What Jev adds

Kronos draws the range. Jev, TypeSafe AI's System One decision model, reads the numbers and returns typed, calibrated calls: up / flat / down with probabilities, a 0 to 3 trust score for the forecast, the price regime, and whether price action agrees with Kronos. It runs through Vercel AI Gateway at $0.042 per million input tokens, so a call costs a fraction of a cent. Jev only sees the snapshot this notebook sends. It has no news feed.

## What Kronos is and is not

Kronos is a research model trained on 12 billion candlesticks from 45 exchanges. It learned the *shape* of price moves. It knows nothing about news, earnings, rates or the Fed. It gives a range with uncertainty, not a target. Treat it as one input, never as a signal to trade on by itself.

Model: [shiyu-coder/Kronos](https://github.com/shiyu-coder/Kronos) (MIT license). Data: Yahoo Finance via yfinance. Jev: [TypeSafe AI](https://docs.typesafe.ai) via [Vercel AI Gateway](https://vercel.com/ai-gateway/models/jev).
